In [1]:
import time
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

# Import all the models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

In [2]:
df = pd.read_csv('../Dataset/WB_Dataset2_without_lon-lat.csv')

X = df.drop('predicted_yield_kg_per_ha', axis=1)
y = df['predicted_yield_kg_per_ha']

# 2. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
categorical_cols = ['state', 'soil_type', 'season', 'crop_name']
numerical_cols = ['rainfall_mm', 'area_ha']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ])

In [4]:
models_to_test = [
    ("Linear Regression", LinearRegression()),
    ("Gradient Boosting", GradientBoostingRegressor(n_estimators=100, random_state=42)),
    ("SVR", SVR(C=1.0, epsilon=0.1)) # SVR requires special target scaling (see below)
]

In [5]:
results = []

print(f"{'Model':<20} | {'R2 Score':<10} | {'MSE':<12} | {'Time (sec)':<10}")
print("-" * 60)

for name, model_obj in models_to_test:
    
    # Special Handling: SVR needs the Target (Y) to be scaled too
    if name == "SVR":
        # Create base pipeline
        p = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model_obj)])
        # Wrap in TransformedTargetRegressor to handle Y scaling automatically
        final_model = TransformedTargetRegressor(regressor=p, transformer=StandardScaler())
    else:
        # Standard Pipeline for all other models
        final_model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model_obj)])
    
    # Train the model
    final_model.fit(X_train, y_train)
    
    # Measure Prediction Time
    start_time = time.time()
    y_pred = final_model.predict(X_test)
    end_time = time.time()
    
    pred_time = end_time - start_time
    
    # Calculate Metrics
    r2 = r2_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    # Store results
    results.append({
        "Model": name,
        "R2 Score": r2,
        "Prediction Time": pred_time
    })
    
    print(f"{name:<20} | {r2:<10.4f} | {mse:<12.0f} | {pred_time:<10.6f}")

Model                | R2 Score   | MSE          | Time (sec)
------------------------------------------------------------
Linear Regression    | 0.9745     | 10037058     | 0.030652  
Gradient Boosting    | 0.9913     | 3412956      | 0.008606  
SVR                  | 0.9777     | 8778653      | 0.081023  


In [6]:
results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
print("\n--- Final Ranking (Best to Worst by Accuracy) ---")
print(results_df)


--- Final Ranking (Best to Worst by Accuracy) ---
               Model  R2 Score  Prediction Time
1  Gradient Boosting  0.991325         0.008606
2                SVR  0.977686         0.081023
0  Linear Regression  0.974487         0.030652
